# BLEU-1


In [ ]:
import jieba
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


# 计算 BLEU 分数（带平滑）
def calculate_bleu_smooth(generated, references, n=1):
    """
    计算 BLEU-N 分数（带平滑）
    :param generated: 生成文本（字符串或列表）
    :param references: 参考文本（字符串或列表）
    :param n: BLEU-N 的 n 值（默认为 1）
    :return: BLEU-N 分数
    """
    # 统一格式：将字符串转换为列表
    if isinstance(generated, str):
        generated = [generated]
    if isinstance(references, str):
        references = [references]

    # 对生成文本和参考文本分词
    generated_tokens = list(jieba.cut(generated[0]))  # 直接使用 jieba 分词
    reference_tokens_list = [
        list(jieba.cut(ref)) for ref in references
    ]  # 对参考文本分词

    # 设置权重（BLEU-N）
    weights = [0] * n
    weights[-1] = 1  # 设置第 N 个权重为 1

    # 使用平滑技术
    smoothing = SmoothingFunction().method1  # 选择平滑方法

    # 计算 BLEU 分数
    bleu_score = sentence_bleu(
        reference_tokens_list,
        generated_tokens,
        weights=weights,
        smoothing_function=smoothing,
    )
    return bleu_score


# 示例数据（字符串格式）
generated_answer = "我喜欢吃苹果"
reference_answers = "我不爱吃雪梨和香蕉"

# 计算 BLEU-1 到 BLEU-4（带平滑）
for n in range(1, 5):
    score = calculate_bleu_smooth(generated_answer, reference_answers, n)
    print(f"BLEU-{n} 分数（带平滑）:", score)

# ROUGE

“f” 表示 f1_score, “p” 表示 precision, “r” 表示 recall


In [ ]:
from rouge_chinese import Rouge
import jieba  # you can use any other word cutting library

hypothesis = "我喜欢吃苹果"
hypothesis = " ".join(jieba.cut(hypothesis))

reference = "我不爱吃雪梨和香蕉"
reference = " ".join(jieba.cut(reference))

rouge = Rouge()
scores = rouge.get_scores(hypothesis, reference)
scores

# llamaindex evaluate


In [ ]:
import chromadb
import chromadb
from llama_index.core import (
    VectorStoreIndex,
    get_response_synthesizer,
    Settings,
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.vector_stores.chroma import ChromaVectorStore
import time
import sys

Settings.embed_model = OllamaEmbedding(model_name="yxl/m3e:latest")


# 创建 Ollama 客户端，增加超时时间和重试次数
def create_ollama_client(model_name, max_retries=3):
    for i in range(max_retries):
        try:
            return Ollama(
                model=model_name,
                request_timeout=300.0,  # 增加超时时间到300秒
                temperature=0.7,
                context_window=4096,
            )
        except Exception as e:
            if i == max_retries - 1:
                print(f"无法创建Ollama客户端: {e}")
                sys.exit(1)
            time.sleep(2)  # 重试前等待


# 创建 Ollama 客户端，两个模型分别用于检索和回答生成
model1_client = Ollama(model="qwen2.5:0.5b", request_timeout=260.0)  # 检索模型
model2_client = Ollama(model="deepseek-r1:1.5b", request_timeout=260.0)  # 回答生成模型


# 连接到 Chroma 数据库
client = chromadb.Client()

# 初始化 Chroma 客户端，指定数据存储路径为当前目录下的 chroma_db 文件夹
db = chromadb.PersistentClient(path="C:/Users/Admin/Desktop/Data/diabetes/chroma_db")

# 获取或创建名为 "quickstart" 的集合，如果该集合不存在，则创建它
chroma_collection = db.get_or_create_collection("quickstart")

# 使用上述集合创建一个 ChromaVectorStore 实例，以便 llama_index 可以与 Chroma 集合进行交互
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# 创建一个存储上下文，指定向量存储为刚刚创建的 ChromaVectorStore 实例
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 从向量存储创建索引
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)


retriever = VectorIndexRetriever(
    index=index,
    llm=model1_client,
    similarity_top_k=3,
)
response_synthesizer = get_response_synthesizer(
    llm=model2_client,
)
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[],
)

while True:
    question = input("请输入您的问题（输入q退出）：")
    if question.lower() == "q":
        break
    response = query_engine.query(question)
    print(response)